In [ ]:
import torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!nvidia-smi

In [ ]:
import time

In [ ]:
# --- Load Gemma 2 2B ---
local_model_path = "/home/jovyan/shared/models/google--gemma-2-2b-it"

tokenizer_gemma = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_gemma = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="auto"
)
print(f"Gemmaload took {time.time()-t0:.2f}s")
print(f"Gemma 2 2B loaded")

In [ ]:
# hide_output
model = model_gemma
tokenizer = tokenizer_gemma
import pandas as pd
#try I need to buy a toy for my nephew, In AI
input_txt = "Transformers are the"
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(model.device)
iterations = []
n_steps = 8
choices_per_step = 5

with torch.no_grad():
    for step in range(n_steps):
        iteration = dict()
        iteration["Input"] = tokenizer.decode(input_ids[0])
        output = model(input_ids=input_ids)
        # Select logits of the first batch and the last token and apply softmax
        next_token_logits = output.logits[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits.float(), dim=-1)
        sorted_ids = torch.argsort(next_token_probs, dim=-1, descending=True)
        #
        # Store tokens with highest probabilities
        for choice_idx in range(choices_per_step):
            token_id = sorted_ids[choice_idx]
            token_prob = next_token_probs[token_id].cpu().item()
            word = tokenizer.decode(token_id)
            ##print(f"    {choice_idx + 1}. {word!r} ({100 * token_prob:.2f}%)")
            token_choice = (
                f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)")
            iteration[f"Choice {choice_idx+1}"] = token_choice
        # Append predicted next token to input
        print()
        next_id = sorted_ids[0].view(1, 1)   # shape [1, 1]
        input_ids = torch.cat([input_ids, next_id], dim=-1)
      
        iterations.append(iteration)

pd.DataFrame(iterations)

# --------------



We can now do the same thing using the built-in function generate

In [ ]:
input_ids = tokenizer_gemma(input_txt, return_tensors="pt")["input_ids"].to(model.device)



In [ ]:
output = model_gemma.generate(input_ids, max_new_tokens=n_steps, do_sample=False)

In [ ]:
print(tokenizer_gemma.decode(output[0]))

Now try a famous example

In [ ]:
max_length = 300
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(model_gemma.device)
output_greedy = model.generate(input_ids, max_length=max_length,
                               do_sample=False)
print(tokenizer_gemma.decode(output_greedy[0]))


# --------

In [ ]:
# encode context the generation is conditioned on
model_inputs = tokenizer_gemma('I enjoy walking with my cute dog', return_tensors='pt').to(model_gemma.device)

# generate 40 new tokens
greedy_output = model_gemma.generate(**model_inputs, max_new_tokens=200)

print("Output:\n" + 150 * '-')
print(tokenizer_gemma.decode(greedy_output[0], skip_special_tokens=True))


In [ ]:
print(tokenizer_gemma.decode(greedy_output[0]))

In [ ]:
# activate beam search and early_stopping
beam_output = model_gemma.generate(
    **model_inputs,
    max_new_tokens=400,
    num_beams=5,
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer_gemma.decode(beam_output[0], skip_special_tokens=True))


In [ ]:
print(tokenizer_gemma.decode(beam_output[0]))

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np

 We import functional which contains stand-alone functions

In [ ]:
import torch.nn.functional as F

Uses F.log_softmax(logits) that takes logits and applies softmax

In [ ]:
def log_probs_from_logits(logits, labels):
    logp = F.log_softmax(logits, dim=-1)
    logp_label = torch.gather(logp, 2, labels.unsqueeze(2)).squeeze(-1)
    return logp_label


The input to sequnec_logprob is the model and the token ids of a sequence

In [ ]:


def sequence_logprob(model, labels, input_len=0):
    with torch.no_grad():
        output = model(labels)
        logits = output.logits.float()                    # bf16 -> fp32
        log_probs = log_probs_from_logits(logits[:, :-1, :], labels[:, 1:])
        gen = log_probs[:, input_len - 1:]                # generated tokens only
    return gen.sum().item(), gen.shape[1]

The output is the logprob of the entire sequence

In [ ]:
#logp = sequence_logprob(model_gemma, output_greedy, input_len=len(input_ids[0]))

prompt_len = input_ids.shape[1]
print(tokenizer_gemma.decode(beam_output[0], skip_special_tokens=True))
logp_cs, n_cs = sequence_logprob(model_gemma, beam_output, input_len=prompt_len)
print(f"\nlog-prob: {logp_cs:.2f}   tokens: {n_cs}   per token: {logp_cs/n_cs:.3f}")

In [ ]:
output_beam = model.generate(input_ids, max_length=max_length, num_beams=5,
                             do_sample=False)
logp_seq, n_tok = sequence_logprob(model_gemma, output_beam, input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob: {logp_seq:.2f}   tokens: {n_tok}   per token: {logp_seq/n_tok:.3f}")

can try changing max_length above

In [ ]:
max_length = 300
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(model_gemma.device)
output_greedy = model_gemma.generate(input_ids, max_length=max_length,
                               do_sample=False)

logp_seq, n_tok = sequence_logprob(model, output_greedy, input_len=len(input_ids[0]))
print(tokenizer.decode(output_greedy[0]))
print(f"\nlog-prob: {logp_seq:.2f}   tokens: {n_tok}   per token: {logp_seq/n_tok:.3f}")

# Temperature

In [ ]:
output_temp = model_gemma.generate(input_ids, max_length=max_length, do_sample=True,
                            temperature=2.0, top_k=0)
print(tokenizer.decode(output_temp[0]))


In [ ]:
output_temp = model_gemma.generate(input_ids, max_length=max_length, do_sample=True,
                            temperature=0.5,top_k=0)
print(tokenizer.decode(output_temp[0]))


# Sampling Methods

In [ ]:
output_temp = model_gemma.generate(input_ids, max_length=max_length, do_sample=True,
                            top_p=0.90)
print(tokenizer.decode(output_temp[0]))


In [ ]:
output_temp = model_gemma.generate(input_ids, max_length=max_length, do_sample=True,
                            top_k=50)
print(tokenizer.decode(output_temp[0]))


# Contrastive Search


Contrastive search looks back in the token sequence, Gemma2 uses by default a HybridCache which is incompatible with indexing

In [ ]:
from transformers import DynamicCache

In [ ]:
output = model_gemma.generate(input_ids, penalty_alpha=0.6, top_k=4, max_new_tokens=512,past_key_values=DynamicCache(), cache_implementation=None)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(output[0], skip_special_tokens=True))
print("" + 100 * '-')

In [ ]:
logp_cs, n_cs = sequence_logprob(model_gemma, output, input_len=prompt_len)
print(tokenizer.decode(output[0, prompt_len:], skip_special_tokens=True))
print(f"\nlog-prob: {logp_cs:.2f}   tokens: {n_cs}   per token: {logp_cs/n_cs:.3f}")